# RocketPy 6DOF Flight Simulation — Hybrid Rocket

**Vehicle:** SMT033 / L277 Hybrid Motor (N2O + Solid Fuel)
**Flight:** February 13, 2026 — Abu Dhabi, UAE
**Actual apogee:** 3,348 m AGL (Fluctus flight computer)

This notebook runs the complete RocketPy 6DOF trajectory simulation using:
- Flight-reconstructed thrust curve (L246, 5520 Ns)
- Real vehicle parameters (measured mass, CG, geometry)
- Real atmospheric data from launch day (Open-Meteo GFS)
- OpenRocket Cd profiles (Mach-dependent)
- Parachute recovery model (6 ft main, CdS = 5.78 m²)

All data files are embedded — no external downloads needed.

## 1. Setup & Installation

In [ ]:
!pip install rocketpy -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rocketpy import Environment, Rocket, Flight, GenericMotor
import os

print(f'RocketPy loaded successfully')

## 2. Create Data Files

Write the thrust curve and drag coefficient files to disk.
These were reconstructed from flight onboard data and OpenRocket analysis.

In [ ]:
# Flight Reconstructed Thrust Curve (.eng format)
# L246 motor: 5520 Ns total impulse, 22.45 s burn
# Reconstructed from onboard Pc via transfer function F = 14.996*Pc + 39.770

eng_content = """;\ Flight Reconstructed Thrust Curve
; Total Impulse: 5520.6 Ns, Max Thrust: 644.8 N, Avg Thrust: 246.0 N
; Reconstructed from onboard Pc data + HFT3/HFT4 ignition shape
L246 100 1330 0 2.3 9.2 FlightRecon
  0.0000    106.545
  0.0199    161.503
  0.0498    318.778
  0.0796    517.696
  0.0995    611.095
  0.1194    644.808
  0.1493    637.574
  0.1791    619.300
  0.1990    608.940
  0.2488    584.078
  0.2985    559.565
  0.3483    554.368
  0.3980    547.353
  0.8160    491.002
  1.2240    487.703
  1.6320    496.250
  2.0400    511.397
  2.4490    512.146
  2.8570    491.002
  3.2650    453.962
  3.6730    448.713
  4.0810    428.768
  4.4890    419.621
  4.8970    411.523
  5.3050    410.773
  5.7130    406.874
  6.1210    398.176
  6.5290    396.227
  6.9380    389.778
  7.3460    381.231
  7.7540    375.082
  8.1620    313.748
  8.5700    267.260
  8.9780    238.168
  9.3860    225.571
  9.7940    211.325
  10.2020    209.375
  10.6100    200.378
  11.0180    179.683
  11.4260    171.735
  11.8350    164.987
  12.2430    156.439
  12.6510    148.641
  13.0590    141.743
  13.4670    133.945
  13.8750    131.546
  14.2830    125.247
  14.6910    119.849
  15.0990    117.599
  15.5070    112.951
  15.9150    108.602
  16.3240    105.753
  16.7320    102.153
  17.1400    99.454
  17.5480    96.905
  17.9560    92.556
  18.3640    91.206
  18.7720    88.357
  19.1800    85.658
  19.5880    83.858
  19.9960    81.909
  20.4040    79.359
  20.8130    77.110
  21.2210    75.760
  21.6290    73.661
  22.0370    71.112
  22.4450    70.062
  22.4550    0.000
"""

with open('Flight_Reconstructed.eng', 'w') as f:
    f.write(eng_content)
print('Created: Flight_Reconstructed.eng')

In [ ]:
# OpenRocket Cd vs Mach profiles
# Power-on drag (motor burning) and power-off drag (coasting)

poweron_drag = """0.0,0.643
0.02,0.691
0.03,0.656
0.04,0.643
0.05,0.643
0.06,0.643
0.07,0.644
0.08,0.644
0.09,0.644
0.10,0.645
0.12,0.646
0.14,0.647
0.16,0.648
0.18,0.650
0.20,0.651
0.22,0.653
0.24,0.655
0.26,0.658
0.28,0.660
0.30,0.663
0.32,0.665
0.34,0.668
0.36,0.671
0.38,0.675
0.40,0.678
0.42,0.682
0.44,0.686
0.46,0.690
0.48,0.694
0.50,0.699
0.52,0.703
0.54,0.708
0.56,0.713
0.58,0.719
0.60,0.724
0.65,0.740
0.70,0.770
0.75,0.810
0.80,0.870
0.85,0.940
0.90,1.020
0.95,1.080
1.00,1.100
1.05,1.060
1.10,1.000
1.20,0.910
1.30,0.840
1.50,0.750
2.00,0.650
"""

poweroff_drag = """0.0,0.660
0.02,0.705
0.03,0.672
0.04,0.660
0.05,0.660
0.06,0.660
0.07,0.660
0.08,0.661
0.09,0.661
0.10,0.661
0.12,0.662
0.14,0.663
0.16,0.664
0.18,0.666
0.20,0.667
0.22,0.669
0.24,0.671
0.26,0.673
0.28,0.660
0.30,0.663
0.32,0.665
0.34,0.668
0.36,0.671
0.38,0.675
0.40,0.678
0.42,0.682
0.44,0.690
0.46,0.695
0.48,0.700
0.50,0.710
0.52,0.715
0.54,0.722
0.56,0.728
0.58,0.735
0.60,0.742
0.65,0.760
0.70,0.790
0.75,0.835
0.80,0.900
0.85,0.970
0.90,1.050
0.95,1.110
1.00,1.130
1.05,1.090
1.10,1.030
1.20,0.940
1.30,0.870
1.50,0.780
2.00,0.680
"""

with open('poweron_drag.csv', 'w') as f:
    f.write(poweron_drag)
with open('poweroff_drag.csv', 'w') as f:
    f.write(poweroff_drag)
print('Created: poweron_drag.csv, poweroff_drag.csv')

## 3. Environment

Launch site: Al Mirfa, Abu Dhabi, UAE (24.18°N, 53.69°E, 5 m ASL)

Atmospheric data from Open-Meteo GFS reanalysis for Feb 13, 2026.

In [ ]:
env = Environment(
    latitude=24.18133,
    longitude=53.688379,
    elevation=5,
)
env.set_date((2026, 2, 13, 12))

# Real wind and atmosphere from Open-Meteo GFS data
env.set_atmospheric_model(
    type='custom_atmosphere',
    wind_u=[
        (0, 0.07), (10, 0.07), (135, 0.00), (818, -0.45),
        (1542, -0.62), (3164, -0.51), (5854, 12.69),
    ],
    wind_v=[
        (0, -4.00), (10, -4.00), (135, -4.19), (818, -1.46),
        (1542, 1.40), (3164, -0.51), (5854, 4.62),
    ],
    pressure=[
        (0, 101500), (135, 100000), (818, 92500),
        (1542, 85000), (3164, 70000), (5854, 50000),
    ],
    temperature=[
        (0, 302.95), (135, 301.65), (818, 295.25),
        (1542, 288.75), (3164, 281.35), (5854, 264.25),
    ],
)

env.info()

## 4. Motor

SMT033 hybrid motor — N2O oxidizer + solid fuel grain.

| Parameter | Value |
|-----------|-------|
| Designation | L246 (flight reconstructed) |
| Total impulse | 5,520 Ns |
| Burn time | 22.45 s |
| Peak thrust | 644.8 N |
| Avg thrust | 246 N |
| Dry mass | 6.9 kg |
| Propellant mass | 2.42 kg |
| Chamber: 100 mm dia x 1330 mm | Nozzle: 50 mm dia |

In [ ]:
motor = GenericMotor(
    thrust_source='./Flight_Reconstructed.eng',
    burn_time=22.455,
    chamber_radius=0.05,        # 100 mm diameter
    chamber_height=1.33,        # 1330 mm length
    chamber_position=1.33 / 2,  # center of chamber
    propellant_initial_mass=2.42,
    nozzle_radius=0.025,        # 50 mm diameter
    dry_mass=6.9,
    dry_inertia=(0.5, 0.5, 0.01),
    nozzle_position=0.0,
    center_of_dry_mass_position=1.33 / 2,
    coordinate_system_orientation='nozzle_to_combustion_chamber',
)

motor.info()

In [ ]:
# Thrust curve visualization
motor.thrust.plot(lower=0, upper=23)
plt.title('Flight Reconstructed Thrust Curve — L246')
plt.show()

## 5. Rocket

| Parameter | Value |
|-----------|-------|
| Body diameter | 100 mm |
| Total length | 2.600 m |
| Airframe mass | 3.780 kg |
| Total launch mass | 13.10 kg |
| CG (without motor) | 1.869 m from tail |
| Nose | Ogive, 300 mm |
| Fins | 4x trapezoidal, 0.5° cant |
| Parachute | 6 ft main, CdS = 5.78 m² |

In [ ]:
rocket = Rocket(
    radius=0.05,                               # 100 mm body diameter
    mass=3.780,                                 # airframe mass without motor
    inertia=(3.5, 3.5, 0.005),
    power_off_drag='poweroff_drag.csv',          # Cd vs Mach from OpenRocket
    power_on_drag='poweron_drag.csv',
    center_of_mass_without_motor=1.869,          # measured CG
    coordinate_system_orientation='tail_to_nose',
)

# Mount motor at tail
rocket.add_motor(motor, position=0.0)

# Nose cone — ogive, 300 mm long, at top
rocket.add_nose(length=0.3, kind='ogive', position=2.600)

# 4 trapezoidal fins with 0.5° cant
rocket.add_trapezoidal_fins(
    n=4,
    root_chord=0.145,
    tip_chord=0.065,
    span=0.08,
    sweep_length=0.11,
    cant_angle=0.5,
    position=0.145,
)

# Boat tail
rocket.add_tail(
    top_radius=0.05,
    bottom_radius=0.03,
    length=0.055,
    position=0.0,
)

# Rail buttons
rocket.set_rail_buttons(
    upper_button_position=1.80,
    lower_button_position=0.40,
    angular_position=88,
)

print(f'Total launch mass: {rocket.total_mass(0):.2f} kg')

In [ ]:
# Parachute — 6 ft main chute, Cd = 2.2
# CdS = 2.2 * pi * (1.8288/2)^2 = 5.78 m^2

def main_trigger(p, h, y):
    return True if y[5] < 0 else False

rocket.add_parachute(
    name='Main',
    cd_s=2.2 * 3.14159 * (1.8288 / 2) ** 2,  # 5.78 m^2
    trigger=main_trigger,  # deploy at apogee (vertical speed < 0)
    sampling_rate=105,
    lag=1.5,               # 1.5 s deployment lag
    noise=(0, 8.3, 0.5),
)

In [ ]:
# Rocket drawing
rocket.draw()

In [ ]:
rocket.info()

## 6. Flight Simulation

Launch from 7 m rail at 83.5° inclination, heading 90° (East).

In [ ]:
flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=7.0,
    inclination=83.5,   # measured launch angle
    heading=90,
    max_time=600,
    time_overshoot=True,
)

## 7. Results

In [ ]:
# Key results
apogee_agl = flight.apogee - env.elevation
actual_apogee = 3348  # Fluctus flight computer
error = (apogee_agl - actual_apogee) / actual_apogee * 100

print('=' * 60)
print('SIMULATION RESULTS vs ACTUAL FLIGHT')
print('=' * 60)
print(f'  Simulated apogee  : {apogee_agl:.0f} m AGL')
print(f'  Actual apogee     : {actual_apogee} m AGL (Fluctus)')
print(f'  Error             : {error:+.1f}%')
print(f'  Apogee time       : {flight.apogee_time:.1f} s')
print(f'  Max speed         : {flight.max_speed:.1f} m/s')
print(f'  Max Mach          : {flight.max_mach_number:.2f}')
print(f'  Max acceleration  : {flight.max_acceleration:.1f} m/s2')
print(f'  Impact velocity   : {flight.impact_velocity:.1f} m/s')
print(f'  Flight time       : {flight.t_final:.1f} s')

In [ ]:
flight.info()

## 8. Trajectory Plots

In [ ]:
# Altitude vs Time
flight.altitude.plot(lower=0, upper=flight.t_final)
plt.title('Altitude vs Time')
plt.show()

In [ ]:
# Speed vs Time
flight.speed.plot(lower=0, upper=50)
plt.title('Speed vs Time (Ascent)')
plt.show()

In [ ]:
# Acceleration vs Time
flight.acceleration.plot(lower=0, upper=30)
plt.title('Acceleration During Burn')
plt.show()

In [ ]:
# Stability margin
flight.stability_margin.plot(lower=0, upper=30)
plt.title('Stability Margin (calibers)')
plt.show()

In [ ]:
# Full trajectory plots (RocketPy built-in)
flight.plots.trajectory_3d()

## 9. Export Results to CSV

In [ ]:
# Export time series for external comparison
sim_z = np.array(flight.z.source)
sim_spd = np.array(flight.speed.source)
sim_acc = np.array(flight.acceleration.source)

t = sim_z[:, 0]
alt = sim_z[:, 1] - env.elevation
speed = np.interp(t, sim_spd[:, 0], sim_spd[:, 1])
accel = np.interp(t, sim_acc[:, 0], sim_acc[:, 1])

with open('simulation_results.csv', 'w') as f:
    f.write('time_s,altitude_m,speed_ms,acceleration_ms2\n')
    for i in range(len(t)):
        f.write(f'{t[i]:.4f},{alt[i]:.2f},{speed[i]:.2f},{accel[i]:.2f}\n')

print(f'Exported: simulation_results.csv ({len(t)} rows)')

# Download in Colab
try:
    from google.colab import files
    files.download('simulation_results.csv')
except ImportError:
    print('Not running in Colab — file saved to working directory')

## 10. All Flight Plots (RocketPy Built-in)

In [ ]:
flight.all_info()